# MILP Phase Analysis

Summarize the computational phases recorded for the dedicated phase-analysis runs.

The notebook reads only phase-analysis artifacts. It does not rerun any benchmark.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 60)

In [ ]:
def find_dataset_dir(dataset_name):
    candidates = [
        Path.cwd() / dataset_name,
        Path('/kaggle/input') / dataset_name,
        Path('/kaggle/working') / dataset_name,
        Path('/kaggle/input/datasets/mohhbrr') / dataset_name,
    ]
    for candidate in candidates:
        if candidate.exists() and (list(candidate.glob('*.csv')) or list(candidate.glob('*.json'))):
            return candidate
    kaggle_root = Path('/kaggle/input')
    if kaggle_root.exists():
        matches = list(kaggle_root.glob(f'**/{dataset_name}'))
        for match in matches:
            if list(match.glob('*.csv')) or list(match.glob('*.json')):
                return match
    raise FileNotFoundError(f'Could not find the {dataset_name} dataset.')

OPTIMIZATION_DATA_DIR = find_dataset_dir('fixing-binary-vars-experiment')
PHASE_DATA_DIR = find_dataset_dir('phase-analysis')
OPTIMIZATION_CSV_PATHS = sorted(OPTIMIZATION_DATA_DIR.glob('*.csv'))
OPTIMIZATION_JSON_PATHS = sorted(OPTIMIZATION_DATA_DIR.glob('*.json'))
PHASE_CSV_PATHS = sorted(PHASE_DATA_DIR.glob('*.csv'))
PHASE_JSON_PATHS = sorted(PHASE_DATA_DIR.glob('*.json'))
if not PHASE_JSON_PATHS:
    raise FileNotFoundError('No phase-analysis JSON artifacts found.')
print(f'Optimization data: {OPTIMIZATION_DATA_DIR}')
print(f'Phase-analysis data: {PHASE_DATA_DIR}')
[path.name for path in PHASE_JSON_PATHS]

In [ ]:
def load_milp_csv(path):
    stem = path.stem
    configuration = (
        'MILP (stable-binary fixing)'
        if stem.startswith('fix-')
        else 'MILP (no stable-binary fixing)'
    )
    experiment = re.sub(r'^(fix|no-fix)-', '', stem)
    csv = pd.read_csv(path)
    csv['optimizer'] = 'MILP'
    csv['configuration'] = configuration
    csv['experiment'] = experiment
    csv['source_file'] = path.name
    return csv

def load_crown_csv(path):
    stem = path.stem
    match = re.match(r'crown-(.+)-([^-]+)$', stem)
    experiment = match.group(1) if match else stem.removeprefix('crown-')
    profile = match.group(2) if match else 'unknown'
    csv = pd.read_csv(path)
    csv['optimizer'] = 'ABCROWN'
    csv['configuration'] = f'ABCROWN ({profile})'
    csv['experiment'] = experiment
    csv['source_file'] = path.name
    csv['solver_status'] = csv.get('abcrown_status', csv.get('status'))
    return csv

tables = []
for path in OPTIMIZATION_CSV_PATHS:
    if path.name.startswith('crown-'):
        tables.append(load_crown_csv(path))
    elif path.name.startswith(('fix-', 'no-fix-')):
        tables.append(load_milp_csv(path))

results = pd.concat(tables, ignore_index=True)
results['solver_status'] = results.get('solver_status', results['status'])
known_experiment_order = [
    f'{network}-{region}'
    for region in ('3pixel', 'global')
    for network in ('3-100', '2-512', '4-1024')
]
experiment_order = known_experiment_order + [
    value for value in sorted(results['experiment'].unique())
    if value not in known_experiment_order
]
results['experiment'] = pd.Categorical(
    results['experiment'], categories=experiment_order, ordered=True
)
results[['optimizer', 'configuration', 'experiment', 'status']].head()

In [ ]:
def load_milp_debug():
    rows = []
    for path in OPTIMIZATION_JSON_PATHS:
        if not path.name.startswith(('fix-', 'no-fix-')):
            continue
        configuration = (
            'MILP (stable-binary fixing)'
            if path.name.startswith('fix-')
            else 'MILP (no stable-binary fixing)'
        )
        experiment = re.sub(r'^(fix|no-fix)-|\.json$', '', path.name)
        for record in json.loads(path.read_text()):
            for direction, details in record.get('directions', {}).items():
                total = details.get('total', {})
                before = details.get('before_presolve', {})
                after = details.get('after_presolve', {})
                progress = details.get('cplex_progress', {})
                rows.append({
                    'optimizer': 'MILP',
                    'configuration': configuration,
                    'experiment': experiment,
                    'instance_id': record['instance_id'],
                    'direction': direction,
                    'binary_variables_before_presolve': total.get('unfixed_binary_variables', before.get('unfixed_binary_variables')),
                    'after_binary_variables': after.get('binary_variables', after.get('all_binary_variables')),
                    'presolve_time_sec': after.get('time_sec', after.get('presolve_time_sec')),
                    'nodes_processed': progress.get('nodes_processed'),
                    'nodes_remaining': progress.get('nodes_remaining'),
                    'lp_iterations': progress.get('iterations'),
                })
    return pd.DataFrame(rows)

milp_debug = load_milp_debug()
milp_debug['experiment'] = pd.Categorical(
    milp_debug['experiment'], categories=experiment_order, ordered=True
)
milp_debug.head()

## Runtime and verification outcome

In [ ]:
summary = (results.groupby(['experiment', 'configuration'], observed=True)
           .agg(instances=('instance_id', 'nunique'),
                unsat=('status', lambda s: (s == 'unsat').sum()),
                unknown=('status', lambda s: (s == 'unknown').sum()),
                median_runtime_sec=('runtime_sec', 'median'),
                mean_runtime_sec=('runtime_sec', 'mean'),
                max_runtime_sec=('runtime_sec', 'max'))
           .reset_index())
summary.sort_values(['experiment', 'configuration'])

In [ ]:
FIGURE_DIR = Path('/kaggle/working/figures')
if not FIGURE_DIR.parent.exists():
    FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

configuration_order = [
    'MILP (stable-binary fixing)',
    'MILP (no stable-binary fixing)',
]
for experiment in experiment_order:
    experiment_data = results[results['experiment'] == experiment]
    if experiment_data.empty:
        continue
    network, region = experiment.rsplit('-', maxsplit=1)
    network_label = network.replace('-', '*')
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.boxplot(
        data=experiment_data,
        x='configuration',
        y='runtime_sec',
        hue='configuration',
        order=configuration_order,
        hue_order=configuration_order,
        legend=False,
        ax=ax,
    )
    ax.set_yscale('log')
    ax.set_ylabel('Runtime (seconds, log scale)')
    ax.set_xlabel('Stable ReLU binary treatment')
    ax.tick_params(axis='x', labelrotation=12)
    ax.set_title(f'Runtime across optimization configurations: {network_label}, {region}')
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f'runtime-{experiment}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## MILP search behavior

In [ ]:
initial_binary_lookup = {}
for path in OPTIMIZATION_JSON_PATHS:
    experiment = re.sub(r'^no-fix-|\.json$', '', path.name)
    for record in json.loads(path.read_text()):
        for direction, details in record.get('directions', {}).items():
            total = details.get('total', {})
            before = details.get('before_presolve', {})
            initial_binary_lookup[(experiment, record['instance_id'], direction)] = total.get(
                'all_binary_variables', before.get('all_binary_variables')
            )

milp_summary = milp_debug.copy()
milp_summary['initial_binaries'] = [
    initial_binary_lookup.get((str(experiment), instance_id, direction))
    for experiment, instance_id, direction in zip(
        milp_summary['experiment'],
        milp_summary['instance_id'],
        milp_summary['direction'],
    )
]
numeric_metrics = ['initial_binaries', 'binary_variables_before_presolve', 'after_binary_variables', 'nodes_processed', 'lp_iterations', 'nodes_remaining']
milp_summary[numeric_metrics] = milp_summary[numeric_metrics].apply(
    pd.to_numeric, errors='coerce'
)
milp_summary.groupby(['experiment', 'configuration'], observed=True).agg(
    median_initial_binaries=('initial_binaries', 'median'),
    median_binary_variables_before_presolve=('binary_variables_before_presolve', 'median'),
    median_binaries_after_presolve=('after_binary_variables', 'median'),
).reset_index().sort_values(['experiment', 'configuration'])

In [ ]:
phase_names = ['bound_tightening', 'encode', 'solver_setup', 'solve']
phase_rows = []
for path in OPTIMIZATION_JSON_PATHS:
    if not path.name.startswith(('fix-', 'no-fix-')):
        continue
    configuration = (
        'MILP (stable-binary fixing)'
        if path.name.startswith('fix-')
        else 'MILP (no stable-binary fixing)'
    )
    experiment = re.sub(r'^(fix|no-fix)-|\.json$', '', path.name)
    for record in json.loads(path.read_text()):
        phases = record.get('phase_timings_sec', {}).get('phases', {})
        phase_seconds = {phase_name: 0.0 for phase_name in phase_names}
        for phase_name, phase_value in phases.items():
            if phase_name == 'bound_tightening' and isinstance(phase_value, dict):
                phase_seconds[phase_name] += sum(
                    value for value in phase_value.values()
                    if isinstance(value, (int, float))
                )
            elif isinstance(phase_value, dict):
                for nested_name, nested_value in phase_value.items():
                    if nested_name not in phase_names:
                        continue
                    if isinstance(nested_value, dict):
                        phase_seconds[nested_name] += sum(
                            value for value in nested_value.values()
                            if isinstance(value, (int, float))
                        )
                    elif isinstance(nested_value, (int, float)):
                        phase_seconds[nested_name] += nested_value
        phase_rows.append({
            'experiment': experiment,
            'configuration': configuration,
            **phase_seconds,
        })

phase_summary = pd.DataFrame(phase_rows)
phase_summary['experiment'] = pd.Categorical(
    phase_summary['experiment'], categories=experiment_order, ordered=True
)
phase_summary.groupby(['experiment', 'configuration'], observed=True)[phase_names].median().reset_index().sort_values(
    ['experiment', 'configuration']
)

## Phase-cost summary

The debug JSON records bound tightening, encoding, solver setup, and solver wall time for both directions. CPLEX presolve is inside `solve`; `solve_after_presolve` is therefore `solve - presolve`, not an additional cost.

In [ ]:
phase_rows = []
direction_rows = []
for path in PHASE_JSON_PATHS:
    if not path.name.startswith(('fix-', 'no-fix-')):
        continue
    configuration = (
        'MILP (stable-binary fixing)' if path.name.startswith('fix-')
        else 'MILP (no stable-binary fixing)'
    )
    experiment = re.sub(r'^(fix|no-fix)-|\.json$', '', path.name)
    for record in json.loads(path.read_text()):
        phases = record.get('phase_timings_sec', {}).get('phases', {})
        bounds = phases.get('bound_tightening', {})
        bound = sum(v for v in bounds.values() if isinstance(v, (int, float)))
        instance = {
            'experiment': experiment, 'configuration': configuration,
            'instance_id': record.get('instance_id'), 'bound_tightening': bound,
        }
        for direction, details in record.get('directions', {}).items():
            timings = details.get('timings_sec', {})
            solve = timings.get('solve')
            presolve = details.get('after_presolve', {}).get('time_sec')
            direction_rows.append({
                'experiment': experiment, 'configuration': configuration,
                'instance_id': record.get('instance_id'), 'direction': direction,
                'encode': timings.get('encode'), 'solver_setup': timings.get('solver_setup'),
                'solve': solve, 'presolve': presolve,
                'solve_after_presolve': max(solve - presolve, 0)
                if solve is not None and presolve is not None else None,
                'binaries_before_presolve': details.get('before_presolve', {}).get('all_binary_variables'),
                'binaries_after_presolve': details.get('after_presolve', {}).get('binary_variables'),
                'nodes': details.get('cplex_progress', {}).get('nodes_processed'),
                'iterations': details.get('cplex_progress', {}).get('iterations'),
            })
            for phase in ('encode', 'solver_setup', 'solve'):
                instance[phase] = instance.get(phase, 0) + (timings.get(phase) or 0)
            instance['presolve'] = instance.get('presolve', 0) + (presolve or 0)
        instance['solve_after_presolve'] = max(instance.get('solve', 0) - instance.get('presolve', 0), 0)
        instance['total_solve'] = instance.get('solve', 0)
        instance['measured_total'] = sum(instance.get(p, 0) for p in (
            'bound_tightening', 'encode', 'solver_setup', 'solve'))
        phase_rows.append(instance)

phase_instances = pd.DataFrame(phase_rows)
direction_phases = pd.DataFrame(direction_rows)
phase_experiment_order = sorted(phase_instances['experiment'].unique())
for frame in (phase_instances, direction_phases):
    frame['experiment'] = pd.Categorical(
        frame['experiment'], categories=phase_experiment_order, ordered=True)

phase_columns = ['bound_tightening', 'encode', 'solver_setup', 'presolve',
                 'solve_after_presolve', 'total_solve', 'measured_total']
phase_medians = (phase_instances.groupby(['experiment', 'configuration'], observed=True)
                 [phase_columns].median().reset_index()
                 .sort_values(['experiment', 'configuration']))
phase_medians

In [ ]:
solver_numeric = ['presolve', 'solve_after_presolve', 'binaries_before_presolve',
                  'binaries_after_presolve', 'nodes', 'iterations']
direction_phases[solver_numeric] = direction_phases[solver_numeric].apply(
    pd.to_numeric, errors='coerce')

solver_medians = (direction_phases.groupby(
    ['experiment', 'configuration', 'direction'], observed=True)
    [['presolve', 'solve_after_presolve', 'binaries_before_presolve',
      'binaries_after_presolve', 'nodes', 'iterations']]
    .median().reset_index()
    .sort_values(['experiment', 'configuration', 'direction']))
solver_medians

In [ ]:
plot_data = phase_medians.melt(
    id_vars=['experiment', 'configuration'],
    value_vars=['bound_tightening', 'encode', 'solver_setup',
                'presolve', 'solve_after_presolve'],
    var_name='phase', value_name='median_seconds')
plot_data = plot_data[plot_data['median_seconds'] > 0]
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_data, x='experiment', y='median_seconds',
            hue='phase', errorbar=None, ax=ax)
ax.set_yscale('log')
ax.set_ylabel('Median seconds (log scale)')
ax.set_xlabel('Benchmark')
ax.set_title('Median MILP phase cost')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()

## Interpretation

Use the first table for overall phase cost, and the second for directional solver behavior. Presolve is reported separately for diagnosis but is already included in `solve`; the post-presolve value is only a derived remainder.